# Test the Gene Annotation Pipeline

This notebook validates the input reader, local human annotation adapters, and the complete mouse pipeline.

## Step 1 — Configure paths and imports

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.input_output import read_gene_csv
from src.annotate_local_files import annotate_hgnc, annotate_ligand_receptor
from src.pipeline import run_pipeline

input_example_csv = project_root / "data" / "example" / "gene_list_MC38OT1_pos.csv"
annotation_dir = project_root / "data" / "annotation"
output_dir = project_root / "results" / "notebook_test"

print("Input:", input_example_csv)
print("Annotations:", annotation_dir)
print("Output:", output_dir)

## Step 2 — Inspect the mouse input CSV

In [ ]:
input_df = pd.read_csv(input_example_csv)

display(input_df.head())
print("Columns:", list(input_df.columns))

## Step 3 — Test gene-list cleaning

In [ ]:
mouse_genes = read_gene_csv(input_example_csv)

assert len(mouse_genes) == 50
assert len(mouse_genes) == len(set(mouse_genes))
assert all(gene.strip() for gene in mouse_genes)

print(f"Read {len(mouse_genes)} unique mouse genes.")

## Step 4 — Test the local human HGNC and ligand–receptor adapters

These are **human-only local resources**. A mouse list cannot be tested against them offline unless mouse genes have already been converted to human orthologs. Therefore, this step uses a small human gene-symbol list directly.

In [ ]:
human_test_genes = ["TRADD", "TNFRSF1A", "FADD", "CASP8", "IFNGR1"]

hgnc_path = annotation_dir / "gene annotation.txt"
lr_path = annotation_dir / "ligand_receptor.csv"

assert hgnc_path.exists(), f"Missing HGNC file: {hgnc_path}"
assert lr_path.exists(), f"Missing ligand-receptor file: {lr_path}"

hgnc_data = pd.read_csv(hgnc_path, sep="\t", low_memory=False)
lr_data = pd.read_csv(lr_path, low_memory=False)

hgnc_test = annotate_hgnc(human_test_genes, hgnc_data)
lr_test = annotate_ligand_receptor(human_test_genes, lr_data)

assert not hgnc_test.empty
assert "Approved symbol" in hgnc_test.columns
assert set(human_test_genes).issubset(set(hgnc_test["input_gene"]))

print("HGNC rows:", len(hgnc_test))
print("Ligand-receptor rows:", len(lr_test))

## Step 5 — Run a local-only mouse pipeline test

In [ ]:
local_tables = run_pipeline(
    input_csv = input_example_csv,
    organism = "mouse",
    output_dir = output_dir / "local_only",
    gene_column = None,
    annotation_dir = annotation_dir,
    use_remote = False,
)

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in local_tables.items()
]))

display(local_tables["run_errors"])

The empty `hgnc` and `ligand_receptor` tables above are expected for a local-only **mouse** run because human orthologs require the remote orthology step. The error table now explains this explicitly.

## Step 6 — Run the complete remote-enabled mouse pipeline

In [ ]:
remote_tables = run_pipeline(
    input_csv = input_example_csv,
    organism = "mouse",
    output_dir = output_dir / "remote",
    gene_column = None,
    annotation_dir = annotation_dir,
    use_remote = True
)

summary = pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in remote_tables.items()
])
    
display(summary)

## Step 7 — Inspect returned annotation tables

In [ ]:
for name in [
    "gprofiler_ids",
    "gprofiler_enrichment",
    "mygene",
    "mouse_to_human_orthologs",
    "hgnc",
    "ligand_receptor"
]:
    print(f"\n{name}: {len(remote_tables[name])} rows")
    display(remote_tables[name].head())